# Benchmark patch extraction
1. Insert 1B rows sequentially

In [2]:
# ---
# jupyter:
#   jupytext:
#     formats: ipynb,py:light
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.16.4
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# +
import psycopg2
import numpy as np
import io
import time

# --- CONFIG ---
DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=postgres-test"
TABLE_NAME = "proto_1_patch_extraction_benchmark"
TOTAL_ROWS = 10_000_000  # for benchmark, adjust as needed
BATCH_SIZE = 100_000

# --- CONNECT ---
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# --- CREATE TABLE (drop if exists for speed) ---
cur.execute(f"""
DROP TABLE IF EXISTS {TABLE_NAME};
CREATE TABLE {TABLE_NAME} (
    id BIGSERIAL PRIMARY KEY,
    pred INT NOT NULL,
    gt INT NOT NULL,
    x FLOAT NOT NULL,
    y FLOAT NOT NULL
);
""")
conn.commit()

# --- STREAMING FUNCTION ---
def copy_batch(preds, gts, xs, ys):
    buffer = io.StringIO()
    for p, g, x, y in zip(preds, gts, xs, ys):
        buffer.write(f"{p}\t{g}\t{x}\t{y}\n")
    buffer.seek(0)
    cur.copy_from(buffer, TABLE_NAME, columns=("pred","gt","x","y"))
    conn.commit()

# --- BENCHMARK LOOP ---
rows_inserted = 0
start_time = time.time()

while rows_inserted < TOTAL_ROWS:
    # generate random batch
    preds = np.random.randint(0, 1000, BATCH_SIZE)
    gts = np.random.randint(0, 1000, BATCH_SIZE)
    xs = np.random.uniform(0, 100, BATCH_SIZE)
    ys = np.random.uniform(0, 100, BATCH_SIZE)
    
    # copy batch into postgres
    copy_batch(preds, gts, xs, ys)
    
    rows_inserted += BATCH_SIZE
    elapsed = time.time() - start_time
    print(f"Inserted {rows_inserted} rows in {elapsed:.2f}s "
          f"({rows_inserted/elapsed:.0f} rows/sec)")

cur.close()
conn.close()

# -

print(f"Estimated time for 1B rows: {1_000_000_000 / (rows_inserted / elapsed) / 3600:.2f} hours")



Inserted 100000 rows in 0.35s (284141 rows/sec)
Inserted 200000 rows in 0.69s (290452 rows/sec)
Inserted 300000 rows in 1.04s (287425 rows/sec)
Inserted 400000 rows in 1.37s (291505 rows/sec)
Inserted 500000 rows in 1.75s (286005 rows/sec)
Inserted 600000 rows in 2.15s (279521 rows/sec)
Inserted 700000 rows in 2.53s (276289 rows/sec)
Inserted 800000 rows in 2.90s (275856 rows/sec)
Inserted 900000 rows in 3.29s (273858 rows/sec)
Inserted 1000000 rows in 3.67s (272689 rows/sec)
Inserted 1100000 rows in 4.03s (273088 rows/sec)
Inserted 1200000 rows in 4.41s (272414 rows/sec)
Inserted 1300000 rows in 4.79s (271174 rows/sec)
Inserted 1400000 rows in 5.16s (271130 rows/sec)
Inserted 1500000 rows in 5.54s (270574 rows/sec)
Inserted 1600000 rows in 5.92s (270333 rows/sec)
Inserted 1700000 rows in 6.28s (270815 rows/sec)
Inserted 1800000 rows in 6.66s (270386 rows/sec)
Inserted 1900000 rows in 7.02s (270573 rows/sec)
Inserted 2000000 rows in 7.37s (271296 rows/sec)
Inserted 2100000 rows in 7.73